<a href="https://colab.research.google.com/github/surya2006-gif/Numerical-Computing/blob/main/Compiler_labfile.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1.lexical Analysis

In [ ]:

import re
import keyword

def lexical_analysis(code):
    tokens = re.findall(
        r'\b\d+\b|[a-zA-Z_]\w*|==|!=|<=|>=|[+\-*/%=<>]|[(){};,]',
        code
    )

    print("TOKEN\t\tTYPE")
    print("----------------------------")

    for token in tokens:
        if keyword.iskeyword(token):
            token_type = "Keyword"
        elif token.isidentifier():
            token_type = "Identifier"
        elif token.isdigit():
            token_type = "Constant"
        elif token in "+-*/%=<>==!=<=>=":
            token_type = "Operator"
        else:
            token_type = "Special Symbol"

        print(token, "\t\t", token_type)


code = """
int a = 10;
int b = 20;
if (a < b) {
    a = a + b;
}
"""

lexical_analysis(code)


TOKEN		TYPE
----------------------------
int 		 Identifier
a 		 Identifier
= 		 Operator
10 		 Constant
; 		 Special Symbol
int 		 Identifier
b 		 Identifier
= 		 Operator
20 		 Constant
; 		 Special Symbol
if 		 Keyword
( 		 Special Symbol
a 		 Identifier
< 		 Operator
b 		 Identifier
) 		 Special Symbol
{ 		 Special Symbol
a 		 Identifier
= 		 Operator
a 		 Identifier
+ 		 Operator
b 		 Identifier
; 		 Special Symbol
} 		 Special Symbol


2.First() and Follow()

In [ ]:

grammar = {
    'E': ['TR'],
    'R': ['+TR', '#'],
    'T': ['FY'],
    'Y': ['*FY', '#'],
    'F': ['(E)', 'i']
}

first = {}
follow = {}

for nt in grammar:
    first[nt] = set()
    follow[nt] = set()


def find_first(symbol):
    if symbol not in grammar:
        return {symbol}

    if first[symbol]:
        return first[symbol]

    for production in grammar[symbol]:
        if production == '#':
            first[symbol].add('#')
            continue

        for char in production:
            result = find_first(char)

            first[symbol].update(result - {'#'})

            if '#' not in result:
                break
        else:
            first[symbol].add('#')

    return first[symbol]


def find_follow():
    follow['E'].add('$')

    changed = True

    while changed:
        changed = False

        for nt in grammar:
            for production in grammar[nt]:
                for i, char in enumerate(production):

                    if char in grammar:
                        old_size = len(follow[char])

                        beta = production[i + 1:]

                        if beta:
                            first_beta = set()
                            nullable = True

                            for symbol in beta:
                                result = find_first(symbol)
                                first_beta.update(result - {'#'})

                                if '#' not in result:
                                    nullable = False
                                    break

                            follow[char].update(first_beta)

                            if nullable:
                                follow[char].update(follow[nt])

                        else:
                            follow[char].update(follow[nt])

                        if len(follow[char]) > old_size:
                            changed = True


for nt in grammar:
    find_first(nt)

find_follow()

print("FIRST SETS:")
for nt in grammar:
    print("FIRST(", nt, ") =", first[nt])

print("\nFOLLOW SETS:")
for nt in grammar:
    print("FOLLOW(", nt, ") =", follow[nt])


FIRST SETS:
FIRST( E ) = {'i', '('}
FIRST( R ) = {'#', '+'}
FIRST( T ) = {'i', '('}
FIRST( Y ) = {'*', '#'}
FIRST( F ) = {'i', '('}

FOLLOW SETS:
FOLLOW( E ) = {'$', ')'}
FOLLOW( R ) = {'$', ')'}
FOLLOW( T ) = {')', '+', '$'}
FOLLOW( Y ) = {')', '+', '$'}
FOLLOW( F ) = {')', '+', '*', '$'}


LL(1) parser

In [ ]:

# LL(1) Parser Implementation

# Grammar:
# E  -> TQ
# Q  -> +TQ | #
# T  -> FR
# R  -> *FR | #
# F  -> (E) | i

table = {
    'E': {'i': 'TQ', '(': 'TQ'},
    'Q': {'+': '+TQ', ')': '#', '$': '#'},
    'T': {'i': 'FR', '(': 'FR'},
    'R': {'+': '#', '*': '*FR', ')': '#', '$': '#'},
    'F': {'i': 'i', '(': '(E)'}
}

terminals = ['i', '+', '*', '(', ')', '$']

def ll1_parser(input_string):
    stack = ['$', 'E']
    input_string += '$'

    index = 0

    print("\nSTACK\t\tINPUT\t\tACTION")
    print("-----------------------------------------------")

    while stack:
        top = stack[-1]
        current = input_string[index]

        print(''.join(stack), '\t\t',
              input_string[index:], '\t\t', end='')

        if top == current == '$':
            print("ACCEPTED")
            return

        if top == current:
            stack.pop()
            index += 1
            print("Match", current)

        elif top in table:
            production = table[top].get(current)

            if production is None:
                print("ERROR: String Rejected")
                return

            stack.pop()

            if production != '#':
                for symbol in reversed(production):
                    stack.append(symbol)

            print(top, "->", production)

        else:
            print("ERROR: String Rejected")
            return

    print("REJECTED")


string = input("Enter input string (use i for identifier): ")
ll1_parser(string)


Enter input string (use i for identifier): i+i*i

STACK		INPUT		ACTION
-----------------------------------------------
$E 		 i+i*i$ 		E -> TQ
$QT 		 i+i*i$ 		T -> FR
$QRF 		 i+i*i$ 		F -> i
$QRi 		 i+i*i$ 		Match i
$QR 		 +i*i$ 		R -> #
$Q 		 +i*i$ 		Q -> +TQ
$QT+ 		 +i*i$ 		Match +
$QT 		 i*i$ 		T -> FR
$QRF 		 i*i$ 		F -> i
$QRi 		 i*i$ 		Match i
$QR 		 *i$ 		R -> *FR
$QRF* 		 *i$ 		Match *
$QRF 		 i$ 		F -> i
$QRi 		 i$ 		Match i
$QR 		 $ 		R -> #
$Q 		 $ 		Q -> #
$ 		 $ 		ACCEPTED


Recursive Descent parser

In [ ]:

# Recursive Descent Parser

input_string = input("Enter the input string (use i for identifier): ")

index = 0

def match(char):
    global index

    if index < len(input_string) and input_string[index] == char:
        index += 1
        return True

    return False


# E -> TQ
def E():
    if T():
        if Q():
            return True
    return False


# Q -> +TQ | epsilon
def Q():
    if match('+'):
        if T():
            return Q()
        return False
    return True


# T -> FR
def T():
    if F():
        if R():
            return True
    return False


# R -> *FR | epsilon
def R():
    if match('*'):
        if F():
            return R()
        return False
    return True


# F -> (E) | i
def F():
    if match('i'):
        return True

    elif match('('):
        if E():
            if match(')'):
                return True

    return False


if E() and index == len(input_string):
    print("String Accepted")
else:
    print("String Rejected")


Enter the input string (use i for identifier): i+i*i
String Accepted


LR(0) PARSER

In [ ]:

# Program to Check LR(0) Grammar

grammar = {
    'S': ['CC'],
    'C': ['cC', 'd']
}

productions = [("S'", "S")]

for nt in grammar:
    for rhs in grammar[nt]:
        productions.append((nt, rhs))

non_terminals = set(grammar.keys()) | {"S'"}
terminals = set()

for lhs, rhs in productions:
    for ch in rhs:
        if ch not in non_terminals:
            terminals.add(ch)

terminals.add('$')


# Closure function
def closure(items):
    items = set(items)

    changed = True
    while changed:
        changed = False

        for p, dot in list(items):
            lhs, rhs = productions[p]

            if dot < len(rhs):
                symbol = rhs[dot]

                if symbol in non_terminals:
                    for i, (l, r) in enumerate(productions):
                        if l == symbol and (i, 0) not in items:
                            items.add((i, 0))
                            changed = True

    return frozenset(items)


# GOTO function
def goto(items, symbol):
    new_items = set()

    for p, dot in items:
        lhs, rhs = productions[p]

        if dot < len(rhs) and rhs[dot] == symbol:
            new_items.add((p, dot + 1))

    if new_items:
        return closure(new_items)

    return frozenset()


# Generate LR(0) states
states = [closure({(0, 0)})]
transitions = {}

i = 0

while i < len(states):
    for symbol in sorted(terminals | non_terminals - {'$', "S'"}):
        new_state = goto(states[i], symbol)

        if new_state:
            if new_state not in states:
                states.append(new_state)

            transitions[(i, symbol)] = states.index(new_state)

    i += 1


# Display LR(0) items
print("LR(0) ITEM SETS")

for i, state in enumerate(states):
    print("\nI", i)

    for p, dot in sorted(state):
        lhs, rhs = productions[p]
        item = rhs[:dot] + "." + rhs[dot:]
        print(lhs, "->", item)


# Construct ACTION and GOTO tables
action = {}
goto_table = {}

for i in range(len(states)):
    for t in terminals:
        action[(i, t)] = ''

    for nt in grammar:
        goto_table[(i, nt)] = ''

conflict = False

def add_action(key, value):
    global conflict

    if action[key] != '' and action[key] != value:
        conflict = True
    else:
        action[key] = value


for i, state in enumerate(states):

    for symbol in terminals - {'$'}:
        if (i, symbol) in transitions:
            add_action((i, symbol),
                       'S' + str(transitions[(i, symbol)]))

    for nt in grammar:
        if (i, nt) in transitions:
            goto_table[(i, nt)] = transitions[(i, nt)]

    for p, dot in state:
        lhs, rhs = productions[p]

        if dot == len(rhs):
            if p == 0:
                add_action((i, '$'), 'ACCEPT')
            else:
                for t in terminals:
                    add_action((i, t), 'R' + str(p))


# Display ACTION table
print("\nACTION TABLE")

for i in range(len(states)):
    print("State", i, end=" : ")

    for t in sorted(terminals):
        print(t, "=", action[(i, t)], end="  ")

    print()


# Display GOTO table
print("\nGOTO TABLE")

for i in range(len(states)):
    print("State", i, end=" : ")

    for nt in grammar:
        print(nt, "=", goto_table[(i, nt)], end="  ")

    print()


# Final result
if conflict:
    print("\nGrammar is NOT LR(0)")
else:
    print("\nGrammar IS LR(0)")


LR(0) ITEM SETS

I 0
S' -> .S
S -> .CC
C -> .cC
C -> .d

I 1
S -> C.C
C -> .cC
C -> .d

I 2
S' -> S.

I 3
C -> .cC
C -> c.C
C -> .d

I 4
C -> d.

I 5
S -> CC.

I 6
C -> cC.

ACTION TABLE
State 0 : $ =   c = S3  d = S4  
State 1 : $ =   c = S3  d = S4  
State 2 : $ = ACCEPT  c =   d =   
State 3 : $ =   c = S3  d = S4  
State 4 : $ = R3  c = R3  d = R3  
State 5 : $ = R1  c = R1  d = R1  
State 6 : $ = R2  c = R2  d = R2  

GOTO TABLE
State 0 : S = 2  C = 1  
State 1 : S =   C = 5  
State 2 : S =   C =   
State 3 : S =   C = 6  
State 4 : S =   C =   
State 5 : S =   C =   
State 6 : S =   C =   

Grammar IS LR(0)


SLR(1) PARSER

In [ ]:

# Program to Check SLR(1) Grammar

grammar = {
    'S': ['CC'],
    'C': ['cC', 'd']
}

productions = [("S'", "S")]

for nt in grammar:
    for rhs in grammar[nt]:
        productions.append((nt, rhs))

non_terminals = set(grammar.keys()) | {"S'"}
terminals = set()

for lhs, rhs in productions:
    for ch in rhs:
        if ch not in non_terminals:
            terminals.add(ch)

terminals.add('$')


# FIRST function
def FIRST(symbol):
    if symbol not in grammar:
        return {symbol}

    result = set()

    for production in grammar[symbol]:
        if production == '#':
            result.add('#')
            continue

        nullable = True

        for ch in production:
            temp = FIRST(ch)
            result.update(temp - {'#'})

            if '#' not in temp:
                nullable = False
                break

        if nullable:
            result.add('#')

    return result


# FOLLOW function
follow = {nt: set() for nt in grammar}
follow['S'].add('$')

changed = True

while changed:
    changed = False

    for lhs, rhs in productions[1:]:
        for i, ch in enumerate(rhs):

            if ch in grammar:
                old = len(follow[ch])
                beta = rhs[i + 1:]

                if beta:
                    nullable = True

                    for symbol in beta:
                        temp = FIRST(symbol)
                        follow[ch].update(temp - {'#'})

                        if '#' not in temp:
                            nullable = False
                            break

                    if nullable:
                        follow[ch].update(follow[lhs])
                else:
                    follow[ch].update(follow[lhs])

                if len(follow[ch]) > old:
                    changed = True


# Closure function
def closure(items):
    items = set(items)

    changed = True

    while changed:
        changed = False

        for p, dot in list(items):
            lhs, rhs = productions[p]

            if dot < len(rhs):
                symbol = rhs[dot]

                if symbol in non_terminals:
                    for i, (l, r) in enumerate(productions):
                        if l == symbol and (i, 0) not in items:
                            items.add((i, 0))
                            changed = True

    return frozenset(items)


# GOTO function
def goto(items, symbol):
    new_items = set()

    for p, dot in items:
        lhs, rhs = productions[p]

        if dot < len(rhs) and rhs[dot] == symbol:
            new_items.add((p, dot + 1))

    return closure(new_items) if new_items else frozenset()


# Generate LR(0) states
states = [closure({(0, 0)})]
transitions = {}

i = 0

while i < len(states):
    for symbol in sorted((terminals | non_terminals) - {'$', "S'"}):
        new_state = goto(states[i], symbol)

        if new_state:
            if new_state not in states:
                states.append(new_state)

            transitions[(i, symbol)] = states.index(new_state)

    i += 1


# Construct SLR parsing table
action = {}
goto_table = {}

for i in range(len(states)):
    for t in terminals:
        action[(i, t)] = ''

    for nt in grammar:
        goto_table[(i, nt)] = ''

conflict = False


def add_action(key, value):
    global conflict

    if action[key] != '' and action[key] != value:
        conflict = True
    else:
        action[key] = value


for i, state in enumerate(states):

    for t in terminals - {'$'}:
        if (i, t) in transitions:
            add_action((i, t), 'S' + str(transitions[(i, t)]))

    for nt in grammar:
        if (i, nt) in transitions:
            goto_table[(i, nt)] = transitions[(i, nt)]

    for p, dot in state:
        lhs, rhs = productions[p]

        if dot == len(rhs):

            if p == 0:
                add_action((i, '$'), 'ACCEPT')

            else:
                for t in follow[lhs]:
                    add_action((i, t), 'R' + str(p))


# Display FOLLOW sets
print("FOLLOW SETS")

for nt in grammar:
    print("FOLLOW(", nt, ") =", follow[nt])


# Display SLR states
print("\nLR(0) ITEM SETS")

for i, state in enumerate(states):
    print("\nI", i)

    for p, dot in sorted(state):
        lhs, rhs = productions[p]
        item = rhs[:dot] + "." + rhs[dot:]
        print(lhs, "->", item)


# Display ACTION table
print("\nACTION TABLE")

for i in range(len(states)):
    print("State", i, end=" : ")

    for t in sorted(terminals):
        print(t, "=", action[(i, t)], end="  ")

    print()


# Display GOTO table
print("\nGOTO TABLE")

for i in range(len(states)):
    print("State", i, end=" : ")

    for nt in grammar:
        print(nt, "=", goto_table[(i, nt)], end="  ")

    print()


# Final result
if conflict:
    print("\nGrammar is NOT SLR(1)")
else:
    print("\nGrammar IS SLR(1)")


FOLLOW SETS
FOLLOW( S ) = {'$'}
FOLLOW( C ) = {'c', '$', 'd'}

LR(0) ITEM SETS

I 0
S' -> .S
S -> .CC
C -> .cC
C -> .d

I 1
S -> C.C
C -> .cC
C -> .d

I 2
S' -> S.

I 3
C -> .cC
C -> c.C
C -> .d

I 4
C -> d.

I 5
S -> CC.

I 6
C -> cC.

ACTION TABLE
State 0 : $ =   c = S3  d = S4  
State 1 : $ =   c = S3  d = S4  
State 2 : $ = ACCEPT  c =   d =   
State 3 : $ =   c = S3  d = S4  
State 4 : $ = R3  c = R3  d = R3  
State 5 : $ = R1  c =   d =   
State 6 : $ = R2  c = R2  d = R2  

GOTO TABLE
State 0 : S = 2  C = 1  
State 1 : S =   C = 5  
State 2 : S =   C =   
State 3 : S =   C = 6  
State 4 : S =   C =   
State 5 : S =   C =   
State 6 : S =   C =   

Grammar IS SLR(1)


CLR PARSER

In [ ]:

# Program to Check CLR(1) Grammar

grammar = {
    'S': ['CC'],
    'C': ['cC', 'd']
}

productions = [("S'", "S")]

for nt in grammar:
    for rhs in grammar[nt]:
        productions.append((nt, rhs))

non_terminals = set(grammar) | {"S'"}
terminals = set()

for lhs, rhs in productions:
    for ch in rhs:
        if ch not in non_terminals:
            terminals.add(ch)

terminals.add('$')


# FIRST function
def FIRST(seq):
    result = set()

    if not seq:
        return {'#'}

    for symbol in seq:
        if symbol not in non_terminals:
            result.add(symbol)
            return result

        nullable = False

        for production in grammar[symbol]:
            if production == '#':
                nullable = True
            else:
                result.update(FIRST(production) - {'#'})
                if '#' in FIRST(production):
                    nullable = True

        if not nullable:
            return result

    result.add('#')
    return result


# Closure function for LR(1)
def closure(items):
    items = set(items)

    changed = True

    while changed:
        changed = False

        for p, dot, lookahead in list(items):
            lhs, rhs = productions[p]

            if dot < len(rhs):
                B = rhs[dot]

                if B in non_terminals:
                    beta = rhs[dot + 1:] + lookahead
                    lookaheads = FIRST(beta) - {'#'}

                    for i, (l, r) in enumerate(productions):
                        if l == B:
                            for a in lookaheads:
                                item = (i, 0, a)

                                if item not in items:
                                    items.add(item)
                                    changed = True

    return frozenset(items)


# GOTO function
def goto(items, symbol):
    new_items = set()

    for p, dot, lookahead in items:
        lhs, rhs = productions[p]

        if dot < len(rhs) and rhs[dot] == symbol:
            new_items.add((p, dot + 1, lookahead))

    return closure(new_items) if new_items else frozenset()


# Generate CLR(1) states
states = [closure({(0, 0, '$')})]
transitions = {}

i = 0

while i < len(states):
    symbols = sorted((terminals | non_terminals) - {'$'})

    for symbol in symbols:
        new_state = goto(states[i], symbol)

        if new_state:
            if new_state not in states:
                states.append(new_state)

            transitions[(i, symbol)] = states.index(new_state)

    i += 1


# Construct ACTION and GOTO tables
action = {}
goto_table = {}

for i in range(len(states)):
    for t in terminals:
        action[(i, t)] = ''

    for nt in grammar:
        goto_table[(i, nt)] = ''

conflict = False


def add_action(key, value):
    global conflict

    if action[key] != '' and action[key] != value:
        conflict = True
    else:
        action[key] = value


for i, state in enumerate(states):

    for t in terminals - {'$'}:
        if (i, t) in transitions:
            add_action((i, t), 'S' + str(transitions[(i, t)]))

    for nt in grammar:
        if (i, nt) in transitions:
            goto_table[(i, nt)] = transitions[(i, nt)]

    for p, dot, lookahead in state:
        lhs, rhs = productions[p]

        if dot == len(rhs):

            if p == 0:
                add_action((i, '$'), 'ACCEPT')
            else:
                add_action((i, lookahead), 'R' + str(p))


# Display CLR(1) items
print("CLR(1) ITEM SETS")

for i, state in enumerate(states):
    print("\nI", i)

    for p, dot, lookahead in sorted(state):
        lhs, rhs = productions[p]
        item = rhs[:dot] + "." + rhs[dot:]
        print(lhs, "->", item, ",", lookahead)


# Display ACTION table
print("\nACTION TABLE")

for i in range(len(states)):
    print("State", i, end=" : ")

    for t in sorted(terminals):
        print(t, "=", action[(i, t)], end="  ")

    print()


# Display GOTO table
print("\nGOTO TABLE")

for i in range(len(states)):
    print("State", i, end=" : ")

    for nt in grammar:
        print(nt, "=", goto_table[(i, nt)], end="  ")

    print()


# Final result
if conflict:
    print("\nGrammar is NOT CLR(1)")
else:
    print("\nGrammar IS CLR(1)")


CLR(1) ITEM SETS

I 0
S' -> .S , $
S -> .CC , $
C -> .cC , c
C -> .cC , d
C -> .d , c
C -> .d , d

I 1
S -> C.C , $
C -> .cC , $
C -> .d , $

I 2
S' -> S. , $

I 3
C -> .cC , c
C -> .cC , d
C -> c.C , c
C -> c.C , d
C -> .d , c
C -> .d , d

I 4
C -> d. , c
C -> d. , d

I 5
S -> CC. , $

I 6
C -> .cC , $
C -> c.C , $
C -> .d , $

I 7
C -> d. , $

I 8
C -> cC. , c
C -> cC. , d

I 9
C -> cC. , $

ACTION TABLE
State 0 : $ =   c = S3  d = S4  
State 1 : $ =   c = S6  d = S7  
State 2 : $ = ACCEPT  c =   d =   
State 3 : $ =   c = S3  d = S4  
State 4 : $ =   c = R3  d = R3  
State 5 : $ = R1  c =   d =   
State 6 : $ =   c = S6  d = S7  
State 7 : $ = R3  c =   d =   
State 8 : $ =   c = R2  d = R2  
State 9 : $ = R2  c =   d =   

GOTO TABLE
State 0 : S = 2  C = 1  
State 1 : S =   C = 5  
State 2 : S =   C =   
State 3 : S =   C = 8  
State 4 : S =   C =   
State 5 : S =   C =   
State 6 : S =   C = 9  
State 7 : S =   C =   
State 8 : S =   C =   
State 9 : S =   C =   

Grammar IS CLR(1

LALR PARSER

In [ ]:

# Program to Check LALR(1) Grammar

grammar = {
    'S': ['CC'],
    'C': ['cC', 'd']
}

# Augmented grammar
productions = [("S'", "S")]

for nt in grammar:
    for rhs in grammar[nt]:
        productions.append((nt, rhs))

non_terminals = set(grammar) | {"S'"}
terminals = set()

for lhs, rhs in productions:
    for ch in rhs:
        if ch not in non_terminals:
            terminals.add(ch)

terminals.add('$')


# FIRST function
def FIRST(seq):
    result = set()

    if not seq:
        return {'#'}

    for symbol in seq:
        if symbol not in non_terminals:
            result.add(symbol)
            return result

        nullable = False

        for production in grammar[symbol]:
            if production == '#':
                nullable = True
            else:
                temp = FIRST(production)
                result.update(temp - {'#'})

                if '#' in temp:
                    nullable = True

        if not nullable:
            return result

    result.add('#')
    return result


# Closure function
def closure(items):
    items = set(items)

    changed = True

    while changed:
        changed = False

        for p, dot, la in list(items):
            lhs, rhs = productions[p]

            if dot < len(rhs):
                B = rhs[dot]

                if B in non_terminals:
                    beta = rhs[dot + 1:] + la
                    lookaheads = FIRST(beta) - {'#'}

                    for i, (l, r) in enumerate(productions):
                        if l == B:
                            for a in lookaheads:
                                item = (i, 0, a)

                                if item not in items:
                                    items.add(item)
                                    changed = True

    return frozenset(items)


# GOTO function
def goto(items, symbol):
    new_items = set()

    for p, dot, la in items:
        lhs, rhs = productions[p]

        if dot < len(rhs) and rhs[dot] == symbol:
            new_items.add((p, dot + 1, la))

    return closure(new_items) if new_items else frozenset()


# Generate CLR(1) states
states = [closure({(0, 0, '$')})]
transitions = {}

i = 0

while i < len(states):
    symbols = sorted((terminals | non_terminals) - {'$'})

    for symbol in symbols:
        new_state = goto(states[i], symbol)

        if new_state:
            if new_state not in states:
                states.append(new_state)

            transitions[(i, symbol)] = states.index(new_state)

    i += 1


# Merge states with same LR(0) core
def core(state):
    return frozenset((p, dot) for p, dot, la in state)


merged_states = []
core_map = {}
old_to_new = {}

for i, state in enumerate(states):
    c = core(state)

    if c not in core_map:
        core_map[c] = len(merged_states)
        merged_states.append(set())

    new_index = core_map[c]
    old_to_new[i] = new_index

    merged_states[new_index].update(state)

merged_states = [
    frozenset(s) for s in merged_states
]


# Merge transitions
merged_transitions = {}

for (i, symbol), j in transitions.items():
    key = (old_to_new[i], symbol)
    value = old_to_new[j]

    if key in merged_transitions and merged_transitions[key] != value:
        raise ValueError("Invalid merged transition")

    merged_transitions[key] = value


# Construct LALR parsing table
action = {}
goto_table = {}

for i in range(len(merged_states)):
    for t in terminals:
        action[(i, t)] = ''

    for nt in grammar:
        goto_table[(i, nt)] = ''

conflict = False


def add_action(key, value):
    global conflict

    if action[key] != '' and action[key] != value:
        conflict = True
    else:
        action[key] = value


for i, state in enumerate(merged_states):

    for t in terminals - {'$'}:
        if (i, t) in merged_transitions:
            add_action(
                (i, t),
                'S' + str(merged_transitions[(i, t)])
            )

    for nt in grammar:
        if (i, nt) in merged_transitions:
            goto_table[(i, nt)] = merged_transitions[(i, nt)]

    for p, dot, la in state:
        lhs, rhs = productions[p]

        if dot == len(rhs):
            if p == 0:
                add_action((i, '$'), 'ACCEPT')
            else:
                add_action((i, la), 'R' + str(p))


# Display merged item sets
print("LALR(1) ITEM SETS")

for i, state in enumerate(merged_states):
    print("\nI", i)

    for p, dot, la in sorted(state):
        lhs, rhs = productions[p]
        item = rhs[:dot] + "." + rhs[dot:]

        print(lhs, "->", item, ",", la)


# Display ACTION table
print("\nACTION TABLE")

for i in range(len(merged_states)):
    print("State", i, end=" : ")

    for t in sorted(terminals):
        print(t, "=", action[(i, t)], end="  ")

    print()


# Display GOTO table
print("\nGOTO TABLE")

for i in range(len(merged_states)):
    print("State", i, end=" : ")

    for nt in grammar:
        print(nt, "=", goto_table[(i, nt)], end="  ")

    print()


# Final result
if conflict:
    print("\nGrammar is NOT LALR(1)")
else:
    print("\nGrammar IS LALR(1)")


LALR(1) ITEM SETS

I 0
S' -> .S , $
S -> .CC , $
C -> .cC , c
C -> .cC , d
C -> .d , c
C -> .d , d

I 1
S -> C.C , $
C -> .cC , $
C -> .d , $

I 2
S' -> S. , $

I 3
C -> .cC , $
C -> .cC , c
C -> .cC , d
C -> c.C , $
C -> c.C , c
C -> c.C , d
C -> .d , $
C -> .d , c
C -> .d , d

I 4
C -> d. , $
C -> d. , c
C -> d. , d

I 5
S -> CC. , $

I 6
C -> cC. , $
C -> cC. , c
C -> cC. , d

ACTION TABLE
State 0 : $ =   c = S3  d = S4  
State 1 : $ =   c = S3  d = S4  
State 2 : $ = ACCEPT  c =   d =   
State 3 : $ =   c = S3  d = S4  
State 4 : $ = R3  c = R3  d = R3  
State 5 : $ = R1  c =   d =   
State 6 : $ = R2  c = R2  d = R2  

GOTO TABLE
State 0 : S = 2  C = 1  
State 1 : S =   C = 5  
State 2 : S =   C =   
State 3 : S =   C = 6  
State 4 : S =   C =   
State 5 : S =   C =   
State 6 : S =   C =   

Grammar IS LALR(1)
